In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Tải và tiền xử lý dữ liệu CIFAR-10
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# 2. Xây dựng cấu trúc Mạng AutoEncoder (Dạng tích chập - Convolutional Autoencoder)
input_img = layers.Input(shape=(32, 32, 3))

# --- Phần Encoder (Nén) ---
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = layers.MaxPooling2D((2, 2), padding='same')(x) # Kích thước giảm còn (16, 16, 32)
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
encoded = layers.MaxPooling2D((2, 2), padding='same')(x) # Bottleneck: (8, 8, 16)

# --- Phần Decoder (Giải nén) ---
x = layers.Conv2DTranspose(16, (3, 3), strides=2, activation='relu', padding='same')(encoded) # Lên (16, 16, 16)
x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation='relu', padding='same')(x) # Lên (32, 32, 32)
decoded = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x) # Khôi phục (32, 32, 3)

# Khởi tạo mô hình AutoEncoder tổng
autoencoder_cifar = models.Model(input_img, decoded)
autoencoder_cifar.compile(optimizer='adam', loss='mean_squared_error')

# 3. Huấn luyện AutoEncoder (Đầu vào và Đầu ra đều là ảnh X_train)
print("--- Đang huấn luyện AutoEncoder trên CIFAR-10 ---")
autoencoder_cifar.fit(X_train, X_train, epochs=5, batch_size=64, validation_data=(X_test, X_test))

# --- Sử dụng Encoder phục vụ bài toán nhận dạng (Classification) ---
encoder_model = models.Model(input_img, encoded)

# Tạo một classifier nối tiếp sau đặc trưng đã nén
clf_input = layers.Input(shape=(8, 8, 16))
clf_flat = layers.Flatten()(clf_input)
clf_dense = layers.Dense(64, activation='relu')(clf_flat)
clf_out = layers.Dense(10, activation='softmax')(clf_dense)
classifier_model = models.Model(clf_input, clf_out)

# Kết hợp: Ảnh -> Encoder -> Classifier
full_recog_model = models.Model(input_img, classifier_model(encoder_model(input_img)))
full_recog_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\n--- Huấn luyện bộ phân loại dựa trên đặc trưng AutoEncoder ---")
full_recog_model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))

In [ ]:
# 1. Tự cho/Giả lập dữ liệu ảnh Cat (0) và Dog (1) kích thước 200x200x3
X_cat_dog = np.random.rand(400, 200, 200, 3).astype('float32')
y_cat_dog = np.random.randint(0, 2, size=(400, 1))

# 2. Xây dựng AutoEncoder cho ảnh lớn 200x200x3
input_cd = layers.Input(shape=(200, 200, 3))

# Encoder
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(input_cd)
x = layers.MaxPooling2D((2, 2), padding='same')(x) # (100, 100, 16)
x = layers.Conv2D(8, (3, 3), activation='relu', padding='same')(x)
encoded_cd = layers.MaxPooling2D((2, 2), padding='same')(x) # Không gian ẩn (50, 50, 8)

# Decoder
x = layers.Conv2DTranspose(8, (3, 3), strides=2, activation='relu', padding='same')(encoded_cd)
x = layers.Conv2DTranspose(16, (3, 3), strides=2, activation='relu', padding='same')(x)
decoded_cd = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

# Xây dựng mô hình phân loại nhị phân dựa trên đặc trưng ẩn
flat_cd = layers.Flatten()(encoded_cd)
dense_cd = layers.Dense(32, activation='relu')(flat_cd)
out_cd = layers.Dense(1, activation='sigmoid')(dense_cd)

model_cd = models.Model(input_cd, out_cd)
model_cd.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("--- Huấn luyện Nhận dạng Cat/Dog qua AutoEncoder ---")
model_cd.fit(X_cat_dog, y_cat_dog, epochs=5, batch_size=16, validation_split=0.2)

In [ ]:
# 1. Tải dữ liệu Fashion-MNIST có sẵn
(X_train_f, y_train_f), (X_test_f, y_test_f) = tf.keras.datasets.fashion_mnist.load_data()
X_train_f = X_train_f.astype('float32') / 255.0
X_test_f = X_test_f.astype('float32') / 255.0

# Làm phẳng ảnh từ (28, 28) sang 784 để đưa vào mạng Dense
X_train_flat = X_train_f.reshape(-1, 784)
X_test_flat = X_test_f.reshape(-1, 784)

# 2. Định nghĩa cấu trúc Dense Autoencoder
input_f = layers.Input(shape=(784,))
encoded_f = layers.Dense(64, activation='relu')(input_f) # Nén từ 784 xuống 64 phần tử
decoded_f = layers.Dense(784, activation='sigmoid')(encoded_f) # Giải nén về 784

# Mô hình phân loại nhận dạng tích hợp từ không gian ẩn 64
dense_clf = layers.Dense(32, activation='relu')(encoded_f)
out_f = layers.Dense(10, activation='softmax')(dense_clf)

model_fashion_ae = models.Model(input_f, out_f)
model_fashion_ae.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("--- Huấn luyện nhận dạng cấu trúc ảnh trên Fashion-MNIST ---")
model_fashion_ae.fit(X_train_flat, y_train_f, epochs=5, batch_size=64, validation_data=(X_test_flat, y_test_f))

In [ ]:
# 1. Tự tạo dữ liệu khuôn mặt giả lập (200 ảnh Nam gán nhãn 0, 200 ảnh Nữ gán nhãn 1)
X_faces = np.random.rand(400, 64, 64, 3).astype('float32')
y_faces = np.array([0]*200 + [1]*200).reshape(-1, 1)

# 2. Xây dựng Convolutional Autoencoder kết hợp phân loại giới tính
input_face = layers.Input(shape=(64, 64, 3))

# Encoder
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_face)
x = layers.MaxPooling2D((2, 2), padding='same')(x) # (32, 32, 32)
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
encoded_face = layers.MaxPooling2D((2, 2), padding='same')(x) # Không gian nén ẩn (16, 16, 16)

# Đầu ra dự đoán nhị phân (Nam/Nữ)
flat_face = layers.Flatten()(encoded_face)
dense_face = layers.Dense(16, activation='relu')(flat_face)
out_face = layers.Dense(1, activation='sigmoid')(dense_face)

model_gender_ae = models.Model(input_face, out_face)
model_gender_ae.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("--- Huấn luyện phân loại giới tính Nam/Nữ bằng AutoEncoder ---")
model_gender_ae.fit(X_faces, y_faces, epochs=5, batch_size=16, validation_split=0.2)